# Building `fresno_almond_yield.csv` from the raw NASS files

This notebook documents how the final tidy dataset `nass_agcomm_yield_sources/fresno_almond_yield.csv` was built from the raw USDA NASS *California County Agricultural Commissioners' Data* files.

**Source:** one raw CSV per crop year (2002–2022), downloaded from
https://www.nass.usda.gov/Statistics_by_State/California/Publications/AgComm/index.php
and saved in `nass_agcomm_yield_sources/` as `<year>.csv`. Each file contains every county and commodity for that year.

**Goal:** for each year, pull out the single Fresno County almond-nut row (excluding the "Almond Hulls" byproduct row) and keep the harvested acres, yield, and production. Yield is also converted from tons/acre to pounds/acre (`FRESNO_YIELD`).

In [1]:
import pandas as pd

DATA_DIR = "nass_agcomm_yield_sources"

## The two raw formats

NASS changed the file layout in 2021, so the raw files come in two flavors:

- **2002–2020:** 11 columns (year, commodity code, crop name, county code, county, harvested acres, yield, production, price, unit, value). The column *names* drift slightly from year to year and the text cells have stray whitespace, but the column *order* is always the same. The almond row is called `ALMONDS ALL`.
- **2021–2022:** a new 16-column layout. The almond row is called `Almonds, Nuts`.

In [2]:
# Peek at one file of each format
print("--- 2002 (old format) ---")
print(pd.read_csv(f"{DATA_DIR}/2002.csv").columns.tolist())
print("\n--- 2021 (new format) ---")
print(pd.read_csv(f"{DATA_DIR}/2021.csv").columns.tolist())

--- 2002 (old format) ---
['year', ' commodity', ' cropname', ' countycode', ' county', ' acres', ' yield', ' production', ' price_per', ' unit', ' value']

--- 2021 (new format) ---
['Year', 'Current Item Name', 'Current Item Code', 'Legacy Item Name', 'Legacy Commodity Code', 'County', 'County Code', 'Harvested Acres', 'Yield', 'Production', 'Price Per Unit', 'Value', 'Unit', 'Row Type Id', 'Commodities In Group', 'Footnote']


## Step 1 — Extract Fresno almonds from the old-format files (2002–2020)

Because the header names vary across years but the column order doesn't, we assign our own column names positionally, strip the whitespace, and filter to the Fresno `ALMONDS ALL` row.

In [3]:
old_cols = ["YEAR", "COMMODITY_CODE", "CROP_NAME", "COUNTY_CODE", "COUNTY",
            "HARVESTED_ACRES", "YIELD", "PRODUCTION", "PRICE", "UNIT", "VALUE"]

rows = []
for year in range(2002, 2021):
    df = pd.read_csv(f"{DATA_DIR}/{year}.csv", header=0, names=old_cols)
    # strip stray whitespace from the text columns
    for col in ["CROP_NAME", "COUNTY"]:
        df[col] = df[col].astype(str).str.strip()
    match = df[(df["COUNTY"] == "Fresno") & (df["CROP_NAME"] == "ALMONDS ALL")]
    rows.append(match)

old = pd.concat(rows, ignore_index=True)

old_tidy = pd.DataFrame({
    "HARVEST_YEAR": pd.to_numeric(old["YEAR"], errors="coerce").astype(int),
    "COUNTY": "FRESNO",
    "COMMODITY": old["CROP_NAME"],
    "HARVESTED_ACRES": pd.to_numeric(old["HARVESTED_ACRES"], errors="coerce"),
    "YIELD_TONS_PER_ACRE": pd.to_numeric(old["YIELD"], errors="coerce"),
    "PRODUCTION_TONS": pd.to_numeric(old["PRODUCTION"], errors="coerce"),
})

old_tidy

,HARVEST_YEAR,COUNTY,COMMODITY,HARVESTED_ACRES,YIELD_TONS_PER_ACRE,PRODUCTION_TONS
0,2002,FRESNO,ALMONDS ALL,63450.0,1.24,78700.0
1,2003,FRESNO,ALMONDS ALL,65018.0,1.07,69600.0
2,2004,FRESNO,ALMONDS ALL,82700.0,1.04,86000.0
3,2005,FRESNO,ALMONDS ALL,88400.0,0.90,79600.0
4,2006,FRESNO,ALMONDS ALL,99300.0,1.16,115000.0
5,2007,FRESNO,ALMONDS ALL,116700.0,1.08,126000.0
6,2008,FRESNO,ALMONDS ALL,120400.0,1.32,159000.0
7,2009,FRESNO,ALMONDS ALL,121000.0,1.16,140000.0
8,2010,FRESNO,ALMONDS ALL,138000.0,1.23,170000.0
9,2011,FRESNO,ALMONDS ALL,150000.0,1.47,221000.0


## Step 2 — Extract Fresno almonds from the new-format files (2021–2022)

The new files have clean headers, so we can filter on them directly. The almond nut row is `Almonds, Nuts` (there is also an `Almonds, All` group row and an `Almonds, Hulls` byproduct row — we don't want those).

In [4]:
rows = []
for year in [2021, 2022]:
    df = pd.read_csv(f"{DATA_DIR}/{year}.csv")
    match = df[(df["County"] == "Fresno") & (df["Current Item Name"] == "Almonds, Nuts")]
    rows.append(match)

new = pd.concat(rows, ignore_index=True)

new_tidy = pd.DataFrame({
    "HARVEST_YEAR": new["Year"].astype(int),
    "COUNTY": "FRESNO",
    "COMMODITY": new["Current Item Name"],
    "HARVESTED_ACRES": pd.to_numeric(new["Harvested Acres"], errors="coerce"),
    "YIELD_TONS_PER_ACRE": pd.to_numeric(new["Yield"], errors="coerce"),
    "PRODUCTION_TONS": pd.to_numeric(new["Production"], errors="coerce"),
})

new_tidy

,HARVEST_YEAR,COUNTY,COMMODITY,HARVESTED_ACRES,YIELD_TONS_PER_ACRE,PRODUCTION_TONS
0,2021,FRESNO,"Almonds, Nuts",287000.0,1.27,364000.0
1,2022,FRESNO,"Almonds, Nuts",303000.0,1.05,318000.0


## Step 3 — Combine and convert yield to pounds/acre

Stack the two eras, sort by year, and add `FRESNO_YIELD` = yield in **pounds per acre** (tons × 2000). Yield and production are on a shelled-kernel basis, as reported by NASS.

In [5]:
final = pd.concat([old_tidy, new_tidy], ignore_index=True).sort_values("HARVEST_YEAR").reset_index(drop=True)
final["FRESNO_YIELD"] = final["YIELD_TONS_PER_ACRE"] * 2000

final

,HARVEST_YEAR,COUNTY,COMMODITY,HARVESTED_ACRES,YIELD_TONS_PER_ACRE,PRODUCTION_TONS,FRESNO_YIELD
0,2002,FRESNO,ALMONDS ALL,63450.0,1.24,78700.0,2480.0
1,2003,FRESNO,ALMONDS ALL,65018.0,1.07,69600.0,2140.0
2,2004,FRESNO,ALMONDS ALL,82700.0,1.04,86000.0,2080.0
3,2005,FRESNO,ALMONDS ALL,88400.0,0.90,79600.0,1800.0
4,2006,FRESNO,ALMONDS ALL,99300.0,1.16,115000.0,2320.0
5,2007,FRESNO,ALMONDS ALL,116700.0,1.08,126000.0,2160.0
6,2008,FRESNO,ALMONDS ALL,120400.0,1.32,159000.0,2640.0
7,2009,FRESNO,ALMONDS ALL,121000.0,1.16,140000.0,2320.0
8,2010,FRESNO,ALMONDS ALL,138000.0,1.23,170000.0,2460.0
9,2011,FRESNO,ALMONDS ALL,150000.0,1.47,221000.0,2940.0


## Step 4 — Save the final CSV

In [6]:
final.to_csv(f"{DATA_DIR}/fresno_almond_yield.csv", index=False)
print(f"Saved {len(final)} rows ({final['HARVEST_YEAR'].min()}-{final['HARVEST_YEAR'].max()}) to {DATA_DIR}/fresno_almond_yield.csv")

Saved 21 rows (2002-2022) to nass_agcomm_yield_sources/fresno_almond_yield.csv
